## SRP324111 / GSE178225

**paper:** [PMID: 34181694](https://pmc.ncbi.nlm.nih.gov/articles/PMC8270445/) - Tissue-specific transcriptional profiling of plasmacytoid dendritic cells reveals a hyperactivated state in chronic SIV infection, 2021

**date, curator:** 2026-01-27, Sara Carsanaro

**notes**
* sex and age of samples in supplementary table 1
* removed samples infected with SIV and H7N3
* for sooty mangabey the age of sexual maturity in males is 4 years [per awd](https://animaldiversity.org/accounts/Cercocebus_atys/)

### annotation summary
run this after annotation is complete

In [21]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,"PBMC - mDC, myeloid dendritic cells",CL:0000782,myeloid dendritic cell,perfect match
8,PBMC - pDC,CL:0000784,plasmacytoid dendritic cell,perfect match


In [22]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,3 years,MmulDv:0000023,3-year-old mature stage,perfect match
1,21 years,MmulDv:0000041,21-year-old stage,perfect match
2,19 years,MmulDv:0000039,19-year-old stage,perfect match
4,4 years,UBERON:0018241,prime adult stage,missing child term


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP324111"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes
Fetching RunIds and ReadHashes for each library...can take several minutes


### library annnotations

In [6]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX11148768,SRP324111,Illumina HiSeq 3000,SRS9210422,,,MmulDv:0000023,3-year-old mature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384753,"PBMC - mDC, myeloid dendritic cells",3 years,,,perfect match,M,,,9544,,,,,,4-63RYa15-mDC-RM_S97,"SAMN19712347,GSM5384753",3,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques, there is evidence in supplementary table 1 with the DOB to support that this stage should be 3 years old mature instead of 3 years old immature",,,,27/01/2026,Total RNA was purified using the Qiagen miRNeasy Mini Kit cDNA was generated using the Clontech SMARTseq v4 Ultra Low Input kit and libraries were prepared for sequencing using the Illumina HiSeq3000,,,,PBMC,RYa15,,,TRANSCRIPTOMIC,cDNA,"SRR14816639,SRR14816640,SRR14816641","A89C44DF1BFD435AE46C11C375A86601,EF2D8F5A7796ACD5103A789B91E395F7,652CBF31378501D2D08BF11A66B42E29"
1,SRX11148767,SRP324111,Illumina HiSeq 3000,SRS9210423,,,MmulDv:0000041,21-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384752,"PBMC - mDC, myeloid dendritic cells",21 years,,,perfect match,M,,,9544,,,,,,6-2RVp4-mDC-RM_S99,"SAMN19712348,GSM5384752",21,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques",,,,27/01/2026,Total RNA was purified using the Qiagen miRNeasy Mini Kit cDNA was generated using the Clontech SMARTseq v4 Ultra Low Input kit and libraries were prepared for sequencing using the Illumina HiSeq3000,,,,PBMC,RVp4,,,TRANSCRIPTOMIC,cDNA,"SRR14816636,SRR14816637,SRR14816638","BFA25F009009B03A5EB539287094E05B,E651C6144A6FB72A3494B42B283E0F36,1B8E4C0BD40DAAF86FCBEB12C52A7011"
2,SRX11148766,SRP324111,Illumina HiSeq 3000,SRS9210425,,,MmulDv:0000039,19-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384751,"PBMC - mDC, myeloid dendritic cells",19 years,,,perfect match,M,,,9544,,,,,,14-91RQn5-mDC-RM_S107,"SAMN19712318,GSM5384751",19,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques",,,,27/01/2026,Total RNA was purified using the Qiagen miRNeasy Mini Kit cDNA was generated using the Clontech SMARTseq v4 Ultra Low Input kit and libraries were prepared for sequencing using the Illumina HiSeq3000,,,,PBMC,RQn5,,,TRANSCRIPTOMIC,cDNA,"SRR14816633,SRR14816634,SRR14816635","0CB779ED1B9474824477057482CF92A6,F2C621825A7AF3BC4E3D951018784615,878782EE019B67DDB393EE13F4FE8BA5"
3,SRX11148765,SRP324111,Illumina HiSeq 3000,SRS9210421,,,MmulDv:0000041,21-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384750,"PBMC - mDC, myeloid dendritic cells",21 years,,,perfect match,M,,,9544,,,,,,2-57RMt4-mDC-RM_S95,"SAMN19712319,GSM5384750",21,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques",,,,27/01/2026,Total RNA was purified using the Qiagen miRNeasy Mini Kit cDNA was generated using the Clontech SMARTseq v4 Ultra Low Input kit and libraries were prepared for sequencing using the Illumina HiSeq3000,,,,PBMC,RMt4,,,TRANSCRIPTOMIC,cDNA,"SRR14816630,SRR14816631,SRR14816632","07779FDF9819027B13DA12B4F341E46A,F5ECE951A703967CFF91BDA315924DBD,FF843576C75C912C5577E73A402D2598"
4,SRX11148764,SRP324111,Illumina HiSeq 3000,SRS9210424,,,UBERON:0018241,prime adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384749,"PBMC - mDC, myeloid dendritic cells",4 years,,,missing child term,M,,,9531,,,,,,24-28FTb1-mDC-SM_S117,"SAMN19712320,GSM5384749",4,year,,,,,SIV neg,,,,27/01/2026,Total RNA was purified using the Qiagen miRNeasy Mini Kit cDNA was generated usin

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [7]:
unique_sorted(library, "infoOrgan")

['PBMC - mDC, myeloid dendritic cells' 'PBMC - pDC']


In [8]:

# PBMC - mDC, myeloid dendritic cells
library.loc[library["infoOrgan"] == "PBMC - mDC, myeloid dendritic cells", "anatId"] = "CL:0000782"
library.loc[library["infoOrgan"] == "PBMC - mDC, myeloid dendritic cells", "anatName"] = "myeloid dendritic cell"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "PBMC - mDC, myeloid dendritic cells", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "PBMC - mDC, myeloid dendritic cells", "anatBiologicalStatus"] = "not documented"

# PBMC - pDC
library.loc[library["infoOrgan"] == "PBMC - pDC", "anatId"] = "CL:0000784"
library.loc[library["infoOrgan"] == "PBMC - pDC", "anatName"] = "plasmacytoid dendritic cell"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "PBMC - pDC", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "PBMC - pDC", "anatBiologicalStatus"] = "not documented"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX11148768,SRP324111,Illumina HiSeq 3000,SRS9210422,CL:0000782,myeloid dendritic cell,MmulDv:0000023,3-year-old mature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384753,"PBMC - mDC, myeloid dendritic cells",3 years,perfect match,not documented,perfect match,M,,,9544,,,,,,4-63RYa15-mDC-RM_S97,"SAMN19712347,GSM5384753",3,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques, there is evidence in supplementary table 1 with the DOB to support that this stage should be 3 years old mature instead of 3 years old immature",,,,27/01/2026,Total RNA was purified using the Qiagen miRNeasy Mini Kit cDNA was generated using the Clontech SMARTseq v4 Ultra Low Input kit and libraries were prepared for sequencing using the Illumina HiSeq3000,,,,PBMC,RYa15,,,TRANSCRIPTOMIC,cDNA,"SRR14816639,SRR14816640,SRR14816641","A89C44DF1BFD435AE46C11C375A86601,EF2D8F5A7796ACD5103A789B91E395F7,652CBF31378501D2D08BF11A66B42E29"
1,SRX11148767,SRP324111,Illumina HiSeq 3000,SRS9210423,CL:0000782,myeloid dendritic cell,MmulDv:0000041,21-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384752,"PBMC - mDC, myeloid dendritic cells",21 years,perfect match,not documented,perfect match,M,,,9544,,,,,,6-2RVp4-mDC-RM_S99,"SAMN19712348,GSM5384752",21,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques",,,,27/01/2026,Total RNA was purified using the Qiagen miRNeasy Mini Kit cDNA was generated using the Clontech SMARTseq v4 Ultra Low Input kit and libraries were prepared for sequencing using the Illumina HiSeq3000,,,,PBMC,RVp4,,,TRANSCRIPTOMIC,cDNA,"SRR14816636,SRR14816637,SRR14816638","BFA25F009009B03A5EB539287094E05B,E651C6144A6FB72A3494B42B283E0F36,1B8E4C0BD40DAAF86FCBEB12C52A7011"
2,SRX11148766,SRP324111,Illumina HiSeq 3000,SRS9210425,CL:0000782,myeloid dendritic cell,MmulDv:0000039,19-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384751,"PBMC - mDC, myeloid dendritic cells",19 years,perfect match,not documented,perfect match,M,,,9544,,,,,,14-91RQn5-mDC-RM_S107,"SAMN19712318,GSM5384751",19,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques",,,,27/01/2026,Total RNA was purified using the Qiagen miRNeasy Mini Kit cDNA was generated using the Clontech SMARTseq v4 Ultra Low Input kit and libraries were prepared for sequencing using the Illumina HiSeq3000,,,,PBMC,RQn5,,,TRANSCRIPTOMIC,cDNA,"SRR14816633,SRR14816634,SRR14816635","0CB779ED1B9474824477057482CF92A6,F2C621825A7AF3BC4E3D951018784615,878782EE019B67DDB393EE13F4FE8BA5"
3,SRX11148765,SRP324111,Illumina HiSeq 3000,SRS9210421,CL:0000782,myeloid dendritic cell,MmulDv:0000041,21-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384750,"PBMC - mDC, myeloid dendritic cells",21 years,perfect match,not documented,perfect match,M,,,9544,,,,,,2-57RMt4-mDC-RM_S95,"SAMN19712319,GSM5384750",21,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques",,,,27/01/2026,Total RNA was purified using the Qiagen miRNeasy Mini Kit cDNA was generated using the Clontech SMARTseq v4 Ultra Low Input kit and libraries were prepared for sequencing using the Illumina HiSeq3000,,,,PBMC,RMt4,,,TRANSCRIPTOMIC,cDNA,"SRR14816630,SRR14816631,SRR14816632","07779FDF9819027B13DA12B4F341E46A,F5ECE951A703967CFF91BDA315924DBD,FF843576C75C912C5577E73A402D2598"
4,SRX11148764,SRP324111,Illumina HiSeq 3000,SRS9210424,CL:0000782,myeloid dendritic cell,UBERON:0018241,prime adult stage,https://www.ncbi.nlm.nih.gov/geo/quer

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [9]:
unique_sorted(library, "infoStage")

['19 years' '21 years' '3 years' '4 years']


i'm doing these manually because it's a bit easier. 4 year old sooty mangabey are sexually mature. for Rhesus macaque i used the species specific ontology

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [10]:
# making these variables because we use them again in the experiment file
my_protocol = 'SMART-Seq v4 Ultra Low Input'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX11148768,SRP324111,Illumina HiSeq 3000,SRS9210422,CL:0000782,myeloid dendritic cell,MmulDv:0000023,3-year-old mature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384753,"PBMC - mDC, myeloid dendritic cells",3 years,perfect match,not documented,perfect match,M,,,9544,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,4-63RYa15-mDC-RM_S97,"SAMN19712347,GSM5384753",3,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques, there is evidence in supplementary table 1 with the DOB to support that this stage should be 3 years old mature instead of 3 years old immature",,,,27/01/2026,Total RNA was purified using the Qiagen miRNeasy Mini Kit cDNA was generated using the Clontech SMARTseq v4 Ultra Low Input kit and libraries were prepared for sequencing using the Illumina HiSeq3000,,,,PBMC,RYa15,,,TRANSCRIPTOMIC,cDNA,"SRR14816639,SRR14816640,SRR14816641","A89C44DF1BFD435AE46C11C375A86601,EF2D8F5A7796ACD5103A789B91E395F7,652CBF31378501D2D08BF11A66B42E29"
1,SRX11148767,SRP324111,Illumina HiSeq 3000,SRS9210423,CL:0000782,myeloid dendritic cell,MmulDv:0000041,21-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384752,"PBMC - mDC, myeloid dendritic cells",21 years,perfect match,not documented,perfect match,M,,,9544,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,6-2RVp4-mDC-RM_S99,"SAMN19712348,GSM5384752",21,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques",,,,27/01/2026,Total RNA was purified using the Qiagen miRNeasy Mini Kit cDNA was generated using the Clontech SMARTseq v4 Ultra Low Input kit and libraries were prepared for sequencing using the Illumina HiSeq3000,,,,PBMC,RVp4,,,TRANSCRIPTOMIC,cDNA,"SRR14816636,SRR14816637,SRR14816638","BFA25F009009B03A5EB539287094E05B,E651C6144A6FB72A3494B42B283E0F36,1B8E4C0BD40DAAF86FCBEB12C52A7011"
2,SRX11148766,SRP324111,Illumina HiSeq 3000,SRS9210425,CL:0000782,myeloid dendritic cell,MmulDv:0000039,19-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384751,"PBMC - mDC, myeloid dendritic cells",19 years,perfect match,not documented,perfect match,M,,,9544,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,14-91RQn5-mDC-RM_S107,"SAMN19712318,GSM5384751",19,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques",,,,27/01/2026,Total RNA was purified using the Qiagen miRNeasy Mini Kit cDNA was generated using the Clontech SMARTseq v4 Ultra Low Input kit and libraries were prepared for sequencing using the Illumina HiSeq3000,,,,PBMC,RQn5,,,TRANSCRIPTOMIC,cDNA,"SRR14816633,SRR14816634,SRR14816635","0CB779ED1B9474824477057482CF92A6,F2C621825A7AF3BC4E3D951018784615,878782EE019B67DDB393EE13F4FE8BA5"
3,SRX11148765,SRP324111,Illumina HiSeq 3000,SRS9210421,CL:0000782,myeloid dendritic cell,MmulDv:0000041,21-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384750,"PBMC - mDC, myeloid dendritic cells",21 years,perfect match,not documented,perfect match,M,,,9544,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,2-57RMt4-mDC-RM_S95,"SAMN19712319,GSM5384750",21,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques",,,,27/01/2026,Total RNA was purified using the Qiagen miRNeasy Mini Kit cDNA was generated using the Clontech SMARTseq v4 Ultra Low Input kit and libraries were prepared for sequencing using the Illumina HiSeq3000,,,,PBMC,RMt4,,,TRANSCRIPTOMIC,cDNA,"SRR14816630,SRR14816631,SRR14816632","07779FDF9819027B13DA12B4F341E46A,F5ECE951A703967CFF91BDA315924DBD,FF843576C75C912C5

#### globin, replicates

In [11]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [12]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-01-28'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX11148768,SRP324111,Illumina HiSeq 3000,SRS9210422,CL:0000782,myeloid dendritic cell,MmulDv:0000023,3-year-old mature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384753,"PBMC - mDC, myeloid dendritic cells",3 years,perfect match,not documented,perfect match,M,,,9544,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,4-63RYa15-mDC-RM_S97,"SAMN19712347,GSM5384753",3,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques, there is evidence in supplementary table 1 with the DOB to support that this stage should be 3 years old mature instead of 3 years old immature",,,SAC,2026-01-28,Total RNA was purified using the Qiagen miRNeasy Mini Kit cDNA was generated using the Clontech SMARTseq v4 Ultra Low Input kit and libraries were prepared for sequencing using the Illumina HiSeq3000,,,,PBMC,RYa15,,,TRANSCRIPTOMIC,cDNA,"SRR14816639,SRR14816640,SRR14816641","A89C44DF1BFD435AE46C11C375A86601,EF2D8F5A7796ACD5103A789B91E395F7,652CBF31378501D2D08BF11A66B42E29"
1,SRX11148767,SRP324111,Illumina HiSeq 3000,SRS9210423,CL:0000782,myeloid dendritic cell,MmulDv:0000041,21-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384752,"PBMC - mDC, myeloid dendritic cells",21 years,perfect match,not documented,perfect match,M,,,9544,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,6-2RVp4-mDC-RM_S99,"SAMN19712348,GSM5384752",21,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques",,,SAC,2026-01-28,Total RNA was purified using the Qiagen miRNeasy Mini Kit cDNA was generated using the Clontech SMARTseq v4 Ultra Low Input kit and libraries were prepared for sequencing using the Illumina HiSeq3000,,,,PBMC,RVp4,,,TRANSCRIPTOMIC,cDNA,"SRR14816636,SRR14816637,SRR14816638","BFA25F009009B03A5EB539287094E05B,E651C6144A6FB72A3494B42B283E0F36,1B8E4C0BD40DAAF86FCBEB12C52A7011"
2,SRX11148766,SRP324111,Illumina HiSeq 3000,SRS9210425,CL:0000782,myeloid dendritic cell,MmulDv:0000039,19-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384751,"PBMC - mDC, myeloid dendritic cells",19 years,perfect match,not documented,perfect match,M,,,9544,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,14-91RQn5-mDC-RM_S107,"SAMN19712318,GSM5384751",19,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques",,,SAC,2026-01-28,Total RNA was purified using the Qiagen miRNeasy Mini Kit cDNA was generated using the Clontech SMARTseq v4 Ultra Low Input kit and libraries were prepared for sequencing using the Illumina HiSeq3000,,,,PBMC,RQn5,,,TRANSCRIPTOMIC,cDNA,"SRR14816633,SRR14816634,SRR14816635","0CB779ED1B9474824477057482CF92A6,F2C621825A7AF3BC4E3D951018784615,878782EE019B67DDB393EE13F4FE8BA5"
3,SRX11148765,SRP324111,Illumina HiSeq 3000,SRS9210421,CL:0000782,myeloid dendritic cell,MmulDv:0000041,21-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384750,"PBMC - mDC, myeloid dendritic cells",21 years,perfect match,not documented,perfect match,M,,,9544,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,2-57RMt4-mDC-RM_S95,"SAMN19712319,GSM5384750",21,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques",,,SAC,2026-01-28,Total RNA was purified using the Qiagen miRNeasy Mini Kit cDNA was generated using the Clontech SMARTseq v4 Ultra Low Input kit and libraries were prepared for sequencing using the Illumina HiSeq3000,,,,PBMC,RMt4,,,TRANSCRIPTOMIC,cDNA,"SRR14816630,SRR14816631,SRR14816632","07779FDF9819027B13DA12B4F341E46A,F5ECE951A703967CFF91BDA315924DBD,FF843

#### comments

In [ ]:
#library.loc[:,'comment'] = ''

#### save complete file with correct columns

In [13]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX11148768,SRP324111,Illumina HiSeq 3000,SRS9210422,CL:0000782,myeloid dendritic cell,MmulDv:0000023,3-year-old mature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384753,"PBMC - mDC, myeloid dendritic cells",3 years,perfect match,not documented,perfect match,M,,,9544,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,4-63RYa15-mDC-RM_S97,"SAMN19712347,GSM5384753",3,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques, there is evidence in supplementary table 1 with the DOB to support that this stage should be 3 years old mature instead of 3 years old immature",,,SAC,2026-01-28
1,SRX11148767,SRP324111,Illumina HiSeq 3000,SRS9210423,CL:0000782,myeloid dendritic cell,MmulDv:0000041,21-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384752,"PBMC - mDC, myeloid dendritic cells",21 years,perfect match,not documented,perfect match,M,,,9544,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,6-2RVp4-mDC-RM_S99,"SAMN19712348,GSM5384752",21,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques",,,SAC,2026-01-28
2,SRX11148766,SRP324111,Illumina HiSeq 3000,SRS9210425,CL:0000782,myeloid dendritic cell,MmulDv:0000039,19-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384751,"PBMC - mDC, myeloid dendritic cells",19 years,perfect match,not documented,perfect match,M,,,9544,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,14-91RQn5-mDC-RM_S107,"SAMN19712318,GSM5384751",19,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques",,,SAC,2026-01-28
3,SRX11148765,SRP324111,Illumina HiSeq 3000,SRS9210421,CL:0000782,myeloid dendritic cell,MmulDv:0000041,21-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384750,"PBMC - mDC, myeloid dendritic cells",21 years,perfect match,not documented,perfect match,M,,,9544,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,2-57RMt4-mDC-RM_S95,"SAMN19712319,GSM5384750",21,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques",,,SAC,2026-01-28
4,SRX11148764,SRP324111,Illumina HiSeq 3000,SRS9210424,CL:0000782,myeloid dendritic cell,UBERON:0018241,prime adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384749,"PBMC - mDC, myeloid dendritic cells",4 years,perfect match,not documented,missing child term,M,,,9531,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,24-28FTb1-mDC-SM_S117,"SAMN19712320,GSM5384749",4,year,,,,,SIV neg,,,SAC,2026-01-28
5,SRX11148763,SRP324111,Illumina HiSeq 3000,SRS9210420,CL:0000782,myeloid dendritic cell,UBERON:0018241,prime adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384748,"PBMC - mDC, myeloid dendritic cells",4 years,perfect match,not documented,missing child term,M,,,9531,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,22-21FRb1-mDC-SM_S115,"SAMN19712321,GSM5384748",4,year,,,,,SIV neg,,,SAC,2026-01-28
6,SRX11148762,SRP324111,Illumina HiSeq 3000,SRS9210419,CL:0000782,myeloid dendritic cell,UBERON:0018241,prime adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384747,"PBMC - mDC, myeloid dendritic cells",4 years,perfect match,not documented,missing child term,M,,,9531,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,20-19FKb1-mDC-SM_S113,"SAMN19712322,GSM5384747",4,year,,,,,SIV neg,,,SAC,2026-01-28
7,SRX11148761,SRP324111,Illumina HiSeq 3000,SRS9210418,CL:0000782,myeloid dendritic cell,UBERON:0018241,prime adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5384746,"PBMC - mDC, myeloid dendritic cells",4 years,perfect match,not documented,missing child term,M,,,9531,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,18-15FJb1-mDC-SM_S111,"SAMN1971

### experiment annotations

In [14]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP324111,Tissue-specific transcriptional profiling of plasmacytoid dendritic cells reveals a hyperactivated state in chronic SIV infection,"HIV associated immune activation (IA) is associated with increased morbidity in people living with HIV (PLWH) on antiretroviral therapy, and remains a barrier for strategies aimed at reducing the HIV reservoir. The underlying mechanisms of IA have not been definitively elucidated, however, persistent production of Type I IFNs and expression of ISGs is considered to be one of the primary factors. Plasmacytoid DCs (pDCs) are a major producer of Type I IFN during viral infections, and are highly immunomodulatory in acute HIV and SIV infection, however their role in chronic HIV/SIV infection has not been firmly established. Here, we performed a detailed transcriptomic characterization of pDCs in chronic SIV infection in rhesus macaques, and in sooty mangabeys, a natural host non-human primate (NHP) species that undergoes non-pathogenic SIV infection. We also investigated the immunostimulatory capacity of lymph node homing pDCs in chronic SIV infection by contrasting gene expression of pDCs isolated from lymph nodes with those from blood. We observed that pDCs in LNs, but not blood, produced high levels of IFN transcripts, and upregulated gene expression programs consistent with T cell activation and exhaustion. We apply a novel strategy to catalogue uncharacterized surface molecules on pDCs, and identified the lymphoid exhaustion markers TIGIT and LAIR1 as highly expressed in SIV infection. pDCs from SIV-infected sooty mangabeys lacked the activation profile of ISG signatures observed in infected macaques. These data demonstrate that pDCs are a primary producer of Type I IFN in chronic SIV infection. Further, this study demonstrated that pDCs trafficking to LNs persist in a highly activated state well into chronic infection. Collectively, these data identify pDCs as a highly immunomodulatory cell population in chronic SIV infection, and a putative therapeutic target to reduce immune activation. Overall design: RNA-Seq analysis of: (i) SIV+ vs SIV- rhesus macaque and sooty mangabey (n=4) (ii) PBMC vs LN in SIV+ rhesus macaques (n=9) (iii) spleen pDCs from H7N3+ vs H7N3- rhesus macaques (n=3)",SRA,,,,,,GSE178225,PRJNA737767,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [15]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

16

In [16]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP324111,Tissue-specific transcriptional profiling of plasmacytoid dendritic cells reveals a hyperactivated state in chronic SIV infection,"HIV associated immune activation (IA) is associated with increased morbidity in people living with HIV (PLWH) on antiretroviral therapy, and remains a barrier for strategies aimed at reducing the HIV reservoir. The underlying mechanisms of IA have not been definitively elucidated, however, persistent production of Type I IFNs and expression of ISGs is considered to be one of the primary factors. Plasmacytoid DCs (pDCs) are a major producer of Type I IFN during viral infections, and are highly immunomodulatory in acute HIV and SIV infection, however their role in chronic HIV/SIV infection has not been firmly established. Here, we performed a detailed transcriptomic characterization of pDCs in chronic SIV infection in rhesus macaques, and in sooty mangabeys, a natural host non-human primate (NHP) species that undergoes non-pathogenic SIV infection. We also investigated the immunostimulatory capacity of lymph node homing pDCs in chronic SIV infection by contrasting gene expression of pDCs isolated from lymph nodes with those from blood. We observed that pDCs in LNs, but not blood, produced high levels of IFN transcripts, and upregulated gene expression programs consistent with T cell activation and exhaustion. We apply a novel strategy to catalogue uncharacterized surface molecules on pDCs, and identified the lymphoid exhaustion markers TIGIT and LAIR1 as highly expressed in SIV infection. pDCs from SIV-infected sooty mangabeys lacked the activation profile of ISG signatures observed in infected macaques. These data demonstrate that pDCs are a primary producer of Type I IFN in chronic SIV infection. Further, this study demonstrated that pDCs trafficking to LNs persist in a highly activated state well into chronic infection. Collectively, these data identify pDCs as a highly immunomodulatory cell population in chronic SIV infection, and a putative therapeutic target to reduce immune activation. Overall design: RNA-Seq analysis of: (i) SIV+ vs SIV- rhesus macaque and sooty mangabey (n=4) (ii) PBMC vs LN in SIV+ rhesus macaques (n=9) (iii) spleen pDCs from H7N3+ vs H7N3- rhesus macaques (n=3)",SRA,partial,Bgee 1K,16,SMART-Seq v4 Ultra Low Input,full_length,GSE178225,PRJNA737767,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [17]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '34181694'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC8270445/'
experiment.loc[:,'DOI'] = '10.1371/journal.ppat.1009674'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP324111,Tissue-specific transcriptional profiling of plasmacytoid dendritic cells reveals a hyperactivated state in chronic SIV infection,"HIV associated immune activation (IA) is associated with increased morbidity in people living with HIV (PLWH) on antiretroviral therapy, and remains a barrier for strategies aimed at reducing the HIV reservoir. The underlying mechanisms of IA have not been definitively elucidated, however, persistent production of Type I IFNs and expression of ISGs is considered to be one of the primary factors. Plasmacytoid DCs (pDCs) are a major producer of Type I IFN during viral infections, and are highly immunomodulatory in acute HIV and SIV infection, however their role in chronic HIV/SIV infection has not been firmly established. Here, we performed a detailed transcriptomic characterization of pDCs in chronic SIV infection in rhesus macaques, and in sooty mangabeys, a natural host non-human primate (NHP) species that undergoes non-pathogenic SIV infection. We also investigated the immunostimulatory capacity of lymph node homing pDCs in chronic SIV infection by contrasting gene expression of pDCs isolated from lymph nodes with those from blood. We observed that pDCs in LNs, but not blood, produced high levels of IFN transcripts, and upregulated gene expression programs consistent with T cell activation and exhaustion. We apply a novel strategy to catalogue uncharacterized surface molecules on pDCs, and identified the lymphoid exhaustion markers TIGIT and LAIR1 as highly expressed in SIV infection. pDCs from SIV-infected sooty mangabeys lacked the activation profile of ISG signatures observed in infected macaques. These data demonstrate that pDCs are a primary producer of Type I IFN in chronic SIV infection. Further, this study demonstrated that pDCs trafficking to LNs persist in a highly activated state well into chronic infection. Collectively, these data identify pDCs as a highly immunomodulatory cell population in chronic SIV infection, and a putative therapeutic target to reduce immune activation. Overall design: RNA-Seq analysis of: (i) SIV+ vs SIV- rhesus macaque and sooty mangabey (n=4) (ii) PBMC vs LN in SIV+ rhesus macaques (n=9) (iii) spleen pDCs from H7N3+ vs H7N3- rhesus macaques (n=3)",SRA,partial,Bgee 1K,16,SMART-Seq v4 Ultra Low Input,full_length,GSE178225,PRJNA737767,34181694,https://pmc.ncbi.nlm.nih.gov/articles/PMC8270445/,10.1371/journal.ppat.1009674,,


#### comments

In [18]:
experiment.loc[:,'comment'] = 'partical b/c removed infected samples - SIV and H7N3'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP324111,Tissue-specific transcriptional profiling of plasmacytoid dendritic cells reveals a hyperactivated state in chronic SIV infection,"HIV associated immune activation (IA) is associated with increased morbidity in people living with HIV (PLWH) on antiretroviral therapy, and remains a barrier for strategies aimed at reducing the HIV reservoir. The underlying mechanisms of IA have not been definitively elucidated, however, persistent production of Type I IFNs and expression of ISGs is considered to be one of the primary factors. Plasmacytoid DCs (pDCs) are a major producer of Type I IFN during viral infections, and are highly immunomodulatory in acute HIV and SIV infection, however their role in chronic HIV/SIV infection has not been firmly established. Here, we performed a detailed transcriptomic characterization of pDCs in chronic SIV infection in rhesus macaques, and in sooty mangabeys, a natural host non-human primate (NHP) species that undergoes non-pathogenic SIV infection. We also investigated the immunostimulatory capacity of lymph node homing pDCs in chronic SIV infection by contrasting gene expression of pDCs isolated from lymph nodes with those from blood. We observed that pDCs in LNs, but not blood, produced high levels of IFN transcripts, and upregulated gene expression programs consistent with T cell activation and exhaustion. We apply a novel strategy to catalogue uncharacterized surface molecules on pDCs, and identified the lymphoid exhaustion markers TIGIT and LAIR1 as highly expressed in SIV infection. pDCs from SIV-infected sooty mangabeys lacked the activation profile of ISG signatures observed in infected macaques. These data demonstrate that pDCs are a primary producer of Type I IFN in chronic SIV infection. Further, this study demonstrated that pDCs trafficking to LNs persist in a highly activated state well into chronic infection. Collectively, these data identify pDCs as a highly immunomodulatory cell population in chronic SIV infection, and a putative therapeutic target to reduce immune activation. Overall design: RNA-Seq analysis of: (i) SIV+ vs SIV- rhesus macaque and sooty mangabey (n=4) (ii) PBMC vs LN in SIV+ rhesus macaques (n=9) (iii) spleen pDCs from H7N3+ vs H7N3- rhesus macaques (n=3)",SRA,partial,Bgee 1K,16,SMART-Seq v4 Ultra Low Input,full_length,GSE178225,PRJNA737767,34181694,https://pmc.ncbi.nlm.nih.gov/articles/PMC8270445/,10.1371/journal.ppat.1009674,,partical b/c removed infected samples - SIV and H7N3


#### save complete file

In [19]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [20]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

#### to add things here

#### check columns match

In [23]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [24]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
53310,SRX10519594,SRP313647,Illumina HiSeq 2500,SRS8643392,UBERON:0002535,gill,UBERON:0000113,post-juvenile adult stage,,gill,1.5 year (adult),perfect match,not documented,other,,USA,,6565,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,Louisiana_30C_rep2,SAMN18613246,1.5,year,,,,,"PMID:33837581, Oysters were acclimated to expe...",20℃ biological replicate 2,,ANN,2026-01-28
53311,SRX10519593,SRP313647,Illumina HiSeq 2500,SRS8643391,UBERON:0002535,gill,UBERON:0000113,post-juvenile adult stage,,gill,1.5 year (adult),perfect match,not documented,other,,USA,,6565,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,Louisiana_30C_rep1,SAMN18613245,1.5,year,,,,,"PMID:33837581, Oysters were acclimated to expe...",20℃ biological replicate 1,,ANN,2026-01-28
53312,SRX11148768,SRP324111,Illumina HiSeq 3000,SRS9210422,CL:0000782,myeloid dendritic cell,MmulDv:0000023,3-year-old mature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,"PBMC - mDC, myeloid dendritic cells",3 years,perfect match,not documented,perfect match,M,,,9544,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,4-63RYa15-mDC-RM_S97,"SAMN19712347,GSM5384753",3,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques, t...",,,SAC,2026-01-28
53313,SRX11148767,SRP324111,Illumina HiSeq 3000,SRS9210423,CL:0000782,myeloid dendritic cell,MmulDv:0000041,21-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,"PBMC - mDC, myeloid dendritic cells",21 years,perfect match,not documented,perfect match,M,,,9544,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,6-2RVp4-mDC-RM_S99,"SAMN19712348,GSM5384752",21,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques",,,SAC,2026-01-28
53314,SRX11148766,SRP324111,Illumina HiSeq 3000,SRS9210425,CL:0000782,myeloid dendritic cell,MmulDv:0000039,19-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,"PBMC - mDC, myeloid dendritic cells",19 years,perfect match,not documented,perfect match,M,,,9544,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,14-91RQn5-mDC-RM_S107,"SAMN19712318,GSM5384751",19,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques",,,SAC,2026-01-28
53315,SRX11148765,SRP324111,Illumina HiSeq 3000,SRS9210421,CL:0000782,myeloid dendritic cell,MmulDv:0000041,21-year-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,"PBMC - mDC, myeloid dendritic cells",21 years,perfect match,not documented,perfect match,M,,,9544,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,2-57RMt4-mDC-RM_S95,"SAMN19712319,GSM5384750",21,year,,,,,"SIV neg, Indian origin (IO) rhesus macaques",,,SAC,2026-01-28
53316,SRX11148764,SRP324111,Illumina HiSeq 3000,SRS9210424,CL:0000782,myeloid dendritic cell,UBERON:0018241,prime adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,"PBMC - mDC, myeloid dendritic cells",4 years,perfect match,not documented,missing child term,M,,,9531,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,24-28FTb1-mDC-SM_S117,"SAMN19712320,GSM5384749",4,year,,,,,SIV neg,,,SAC,2026-01-28


In [25]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1008,SRP062344,Oncorhynchus kisutch Transcriptome or Gene exp...,"Genomic resources for analysis of coho salmon,...",SRA,total,Bgee 1K,9,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA292718,26614614,,10.1016/j.margen.2015.11.008,,
1009,SRP313647,Transcriptomic signatures of temperature adapt...,"In this study, we used comparative transcripto...",SRA,total,Bgee 1K,24,NEBNext Ultra RNA Library Prep Kit,full_length,,PRJNA719567,33837581,https://academic.oup.com/jeb/article/34/8/1212...,10.1111/jeb.13789,,"oyster, adult stage mapped to UBERON:0000113 ..."
1010,SRP324111,Tissue-specific transcriptional profiling of p...,HIV associated immune activation (IA) is assoc...,SRA,partial,Bgee 1K,16,SMART-Seq v4 Ultra Low Input,full_length,GSE178225,PRJNA737767,34181694,https://pmc.ncbi.nlm.nih.gov/articles/PMC8270445/,10.1371/journal.ppat.1009674,,partical b/c removed infected samples - SIV an...


### add annotations to git

In [26]:
! git pull

Already up to date.


In [27]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [28]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../1_template/1_complete_bulk_template.ipynb
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [29]:
! git add $git_experiment_path $git_library_path

In [30]:
! git commit -m $commit_message_exp

[develop 202053b] adding annotated bulk experiment SRP324111
 2 files changed, 17 insertions(+)


In [31]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 3.53 KiB | 1.17 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   4c2547c..202053b  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push